# GNM Semantic Sampler Demo

### How to use
1. **Categorical Sampling**: Select an expression or a combination of gender/ethnicity from the dropdowns and click **Sample Expression** or **Sample Identity**. This generates a single random variation within that category.
2. **Blending Controls**: Select two distinct categories and adjust the **Mix Slider** to interpolate between them. Click **Apply Blend** to visualize the hybrid result.
3. **Random Exploration**: Click **Random Sample** to generate a completely random identity and expression in one go.
4. **Resetting**: Click **Reset to Template** to return to the neutral average head with a zero-expression state.

### How do I run this?


In the menu-bar above, click **Runtime &rarr; Run all**.

In [ ]:
# @title Import dependencies.
# @test {"skip": true}

from IPython.display import display
import ipywidgets as widgets
import numpy as np
import tensorflow as tf  # Needed to access tf.keras from semantic_sampler.

from gnm.shape import gnm_numpy
from gnm.shape import gnm_jupyter_viewer
from gnm.shape import semantic_sampler
GNMMeshViewer = gnm_jupyter_viewer.GNMMeshViewer



In [ ]:
# @title Instantiate Models

gnm = gnm_numpy.GNM.from_local(
    version=gnm_numpy.GNMMajorVersion.V3,
    variant=gnm_numpy.GNMVariant.HEAD
)
expr_sampler = semantic_sampler.ExpressionSampler()
id_sampler = semantic_sampler.IdentitySampler()


In [ ]:
#@title Define the Simple Sampler GUI

class SimpleSemanticGNMDemoGui:
  def __init__(self, gnm, expr_sampler, id_sampler, update_callback):
    self.gnm = gnm
    self.expr_sampler = expr_sampler
    self.id_sampler = id_sampler
    self.update_callback = update_callback

    # Initial State
    self.current_identity = np.zeros(gnm.identity_dim)
    self.current_expression = np.zeros(gnm.expression_dim)
    self.current_rotations = np.zeros((gnm.num_joints, 3))

    expr_options = [(e.name.lower(), e) for e in semantic_sampler.Expression]
    ethn_options = [(e.name.lower(), e) for e in semantic_sampler.Ethnicity]
    gender_options = [(g.name.lower(), g) for g in semantic_sampler.Gender]

    # --- SECTION 1: Categorical Sampling ---
    self.expr_dropdown = widgets.Dropdown(options=expr_options, value=semantic_sampler.Expression.HAPPY, description='Expr:')
    self.gender_dropdown = widgets.Dropdown(options=gender_options, value=semantic_sampler.Gender.FEMALE, description='Gender:')
    self.ethnicity_dropdown = widgets.Dropdown(options=ethn_options, value=semantic_sampler.Ethnicity.WHITE, description='Ethnicity:')

    self.sample_expr_button = widgets.Button(description='Sample Expression', button_style='primary')
    self.sample_id_button = widgets.Button(description='Sample Identity', button_style='success')

    self.sample_expr_button.on_click(self.on_sample_expression)
    self.sample_id_button.on_click(self.on_sample_identity)

    # --- SECTION 2: Blending ---
    self.expr_blend_1 = widgets.Dropdown(options=expr_options, value=semantic_sampler.Expression.HAPPY, description='Expr 1:')
    self.expr_blend_2 = widgets.Dropdown(options=expr_options, value=semantic_sampler.Expression.SURPRISE, description='Expr 2:')
    self.expr_mix_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description='Mix:')

    self.ethn_blend_1 = widgets.Dropdown(options=ethn_options, value=semantic_sampler.Ethnicity.WHITE, description='Ethn 1:')
    self.ethn_blend_2 = widgets.Dropdown(options=ethn_options, value=semantic_sampler.Ethnicity.BLACK, description='Ethn 2:')
    self.ethn_mix_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description='Mix:')

    self.blend_button = widgets.Button(description='Apply Blend', button_style='info')
    self.blend_reset_button = widgets.Button(description='Reset Mixes', button_style='warning')

    self.blend_button.on_click(self.on_blend)
    self.blend_reset_button.on_click(self.on_blend_reset)

    # --- Global Controls ---
    self.sample_random_button = widgets.Button(description='Random Sample', button_style='info')
    self.reset_button = widgets.Button(description='Reset to Template', button_style='warning')
    self.sample_random_button.on_click(self.on_sample_random)
    self.reset_button.on_click(self.on_reset)

    # --- Layout ---
    cat_ui = widgets.VBox([
        widgets.HTML('<b>Categorical Sampling</b>'),
        widgets.HBox([self.expr_dropdown, self.sample_expr_button]),
        widgets.HBox([self.gender_dropdown, self.ethnicity_dropdown, self.sample_id_button])
    ])

    blend_ui = widgets.VBox([
        widgets.HTML('<b>Blending Controls</b>'),
        widgets.HBox([self.expr_blend_1, self.expr_blend_2, self.expr_mix_slider]),
        widgets.HBox([self.ethn_blend_1, self.ethn_blend_2, self.ethn_mix_slider]),
        widgets.HBox([self.blend_button, self.blend_reset_button], layout=widgets.Layout(gap='20px'))
    ])

    global_ui = widgets.HBox([self.sample_random_button, self.reset_button], layout=widgets.Layout(gap='20px'))

    gui = widgets.VBox([cat_ui, widgets.HTML('<hr>'), blend_ui, widgets.HTML('<hr>'), global_ui])
    display(gui)

  def on_sample_expression(self, _):
    self.current_expression = self.expr_sampler.sample_expression(self.expr_dropdown.value, num_samples=1)[0]
    self.update_callback()

  def on_sample_identity(self, _):
    self.current_identity = self.id_sampler.sample_identity(self.gender_dropdown.value, self.ethnicity_dropdown.value, num_samples=1)[0]
    self.update_callback()

  def on_blend(self, _):
    # Interpolate Expressions
    w2_expr = self.expr_mix_slider.value
    w1_expr = 1.0 - w2_expr
    expr_weights = {self.expr_blend_1.value: w1_expr, self.expr_blend_2.value: w2_expr}
    self.current_expression = self.expr_sampler.blend_expressions(expr_weights)

    # Interpolate Ethnicities (maintain current gender selection)
    w2_ethn = self.ethn_mix_slider.value
    w1_ethn = 1.0 - w2_ethn
    ethn_weights = {self.ethn_blend_1.value: w1_ethn, self.ethn_blend_2.value: w2_ethn}
    gender_weights = {self.gender_dropdown.value: 1.0} # Keep selected gender

    self.current_identity = self.id_sampler.blend_identities(gender_weights, ethn_weights, num_samples=1)[0]
    self.update_callback()

  def on_blend_reset(self, _):
    self.expr_mix_slider.value = 0.5
    self.ethn_mix_slider.value = 0.5

  def on_sample_random(self, _):
    expr = np.random.choice(list(semantic_sampler.Expression))
    self.expr_dropdown.value = expr
    gender = np.random.choice(list(semantic_sampler.Gender))
    self.gender_dropdown.value = gender
    ethnicity = np.random.choice(list(semantic_sampler.Ethnicity))
    self.ethnicity_dropdown.value = ethnicity

    self.current_expression = self.expr_sampler.sample_expression(expr, num_samples=1)[0]
    self.current_identity = self.id_sampler.sample_identity(gender, ethnicity, num_samples=1)[0]
    self.update_callback()

  def on_reset(self, _):
    self.current_identity = np.zeros(self.gnm.identity_dim)
    self.current_expression = np.zeros(self.gnm.expression_dim)
    self.update_callback()

  def get_parameters(self):
    return self.current_identity, self.current_expression, self.current_rotations

In [ ]:
#@title Launch the Demo Sampler

viewer = GNMMeshViewer(gnm)

def trigger_update():
  identity, expression, rotations = gui.get_parameters()
  vertices = gnm(identity, expression, rotations)
  viewer.update(vertices)

gui = SimpleSemanticGNMDemoGui(gnm, expr_sampler, id_sampler, trigger_update)
trigger_update()